In [0]:
# Databricks Notebook: gold_publish.py
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder.getOrCreate()

# Configuration widgets
dbutils.widgets.text("catalog_name", "azure-medallion-university-chapters")
dbutils.widgets.text("base_path", "/Volumes")
dbutils.widgets.text("run_id", "")

# Get configuration values
catalog_name = dbutils.widgets.get("catalog_name")
base_path = dbutils.widgets.get("base_path")
run_id = dbutils.widgets.get("run_id")

# Dynamic path construction
silver_path = f"{base_path}/{catalog_name}/sliver/university_chapters"
gold_path = f"{base_path}/{catalog_name}/gold/university_chapters"

# Retry logic for reading Silver data
max_retries = 3
retry_delay = 5
for attempt in range(max_retries):
    try:
        silver_df = spark.read.parquet(silver_path)
        break
    except Exception as e:
        if attempt < max_retries - 1:
            time.sleep(retry_delay)
        else:
            raise RuntimeError(f"Failed to read silver data after {max_retries} attempts: {e}")

# Gold = consumer-facing dataset (OK + WARNING only)
try:
    gold_df = silver_df.filter(col("dq_status").isin(["OK", "WARNING"]))
except Exception as e:
    raise RuntimeError(f"Schema mismatch or filter error: {e}")

# Retry logic for saving Gold output
for attempt in range(max_retries):
    try:
        gold_df.write.mode("overwrite").parquet(gold_path)
        break
    except Exception as e:
        if attempt < max_retries - 1:
            time.sleep(retry_delay)
        else:
            raise RuntimeError(f"Failed to write gold data after {max_retries} attempts: {e}")

# Log counts
try:
    rows_gold = gold_df.count()
    rows_warned = gold_df.filter(col("dq_status") == "WARNING").count()
    rows_ok = gold_df.filter(col("dq_status") == "OK").count()
except Exception as e:
    raise RuntimeError(f"Error calculating counts: {e}")

print(f"Run {run_id}: Gold published. Total={rows_gold}, Warned={rows_warned}, OK={rows_ok}")